# Chapter 6 — Fine-tuning for classification

Chapter 5 produced a reusable GPT architecture and loaded pretrained OpenAI weights. This chapter adapts those representations to a
binary supervised task: classifying SMS messages as legitimate (`ham`) or unwanted (`spam`).

For a modern instruction-tuned LLM, prompting should normally be evaluated before task-specific fine-tuning. This GPT-2-sized model is
not instruction-tuned, however, and the chapter's main purpose is pedagogical: learning dataset preparation, classification heads,
parameter freezing, supervised loss, and evaluation. A production spam system should still be compared with simpler baselines such as
TF–IDF plus logistic regression.

## 6.1 Downloading the SMS Spam Collection

The UCI dataset contains 5,572 labeled SMS messages in a small ZIP archive. Keep acquisition idempotent: if the final TSV already exists,
skip network and extraction work so rerunning the notebook does not overwrite local data.

In [1]:
import os
import urllib.request
import zipfile
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(
    url: str,
    zip_path: str | Path,
    extracted_path: str | Path,
    data_file_path: Path,
) -> None:
    """Download and extract the UCI SMS Spam Collection when absent.

    Args:
        url: URL of the source ZIP archive.
        zip_path: Local path used for the downloaded archive.
        extracted_path: Directory that receives extracted archive members.
        data_file_path: Final path of the renamed tab-separated dataset.

    Returns:
        None.

    Raises:
        urllib.error.URLError: If the dataset download fails.
        zipfile.BadZipFile: If the downloaded archive is invalid.
        OSError: If writing, extracting, or renaming a file fails.
    """
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # The archive is small enough to download into memory before writing.
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # The URL is a trusted UCI source; extract its members into one directory.
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Give the extensionless source file a .tsv suffix that describes its format.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")


download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction.


## 6.2 Loading messages into a dataframe

The source is tab-separated and has no header. Assign `Label` to the external `ham`/`spam` category and `Text` to the raw message. Each
row is one supervised example; the two columns are its target and input.

In [2]:
import pandas as pd

# df: (num_messages=5_572, num_columns=2)
# The source file has no header, so assign descriptive column names.
df = pd.read_csv(
    data_file_path,
    sep="	",
    header=None,
    names=["Label", "Text"],
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


### Inspecting class balance

The original dataset contains many more ham messages than spam messages. A classifier that favors ham could therefore obtain deceptively
high raw accuracy without learning useful spam detection.

In [3]:
# Count examples before balancing to expose the original class imbalance.
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


## 6.3 Creating a balanced pedagogical dataset

Keep all 747 spam examples and randomly select 747 ham examples. The fixed seed makes the selected subset reproducible.

Downsampling makes accuracy easier to interpret and reduces training cost, but discards most legitimate messages and changes class
prevalence. It is a teaching simplification rather than a universally preferred production strategy.

In [4]:
def create_balanced_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Downsample ham messages to match the number of spam messages.

    Args:
        df: SMS examples with `Label` and `Text` columns.

    Returns:
        Dataframe containing every spam example and an equally sized,
        reproducibly sampled ham subset.
    """
    # num_spam is the minority-class row count: 747 for this dataset.
    num_spam = df[df["Label"] == "spam"].shape[0]
    # Keep a reproducible subset instead of allowing the majority class to dominate.
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam,
        random_state=123,
    )
    # balanced_df: (2 * num_spam, num_columns=2)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


### Encoding class labels

Cross-entropy expects integer class targets rather than strings. Map legitimate messages to `0` and spam messages to `1`; these meanings
must remain consistent through datasets, training, evaluation, and inference.

In [5]:
# Convert external string labels into integer targets for cross-entropy.
# ham -> 0 and spam -> 1; balanced_df remains (num_messages=1_494, 2).
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

## 6.4 Creating training, validation, and test splits

Shuffle once with a fixed seed, then allocate 70% for parameter updates, 10% for model-selection feedback, and the remaining 20% for a
final held-out estimate. The test set must not guide training decisions.

Integer boundaries can cause a one-row rounding difference from the exact percentages. All rows still belong to exactly one split.

In [6]:
def random_split(
    df: pd.DataFrame,
    train_frac: float,
    validation_frac: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Shuffle and partition examples into training, validation, and test sets.

    Args:
        df: Labeled examples to partition.
        train_frac: Fraction assigned to the training set.
        validation_frac: Fraction assigned to the validation set. The remaining
            fraction becomes the test set.

    Returns:
        Training, validation, and test dataframes in that order.
    """
    # Shuffle reproducibly so the concatenated classes are mixed before slicing.
    shuffled_df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(shuffled_df) * train_frac)
    validation_end = train_end + int(len(shuffled_df) * validation_frac)

    # With 70%/10%, the unspecificed final 20% becomes the held-out test set.
    train_df = shuffled_df[:train_end]
    validation_df = shuffled_df[train_end:validation_end]
    test_df = shuffled_df[validation_end:]

    return train_df, validation_df, test_df


train_df, validation_df, test_df = random_split(
    balanced_df,
    train_frac=0.7,
    validation_frac=0.1,
)
train_df.shape, validation_df.shape, test_df.shape

((1045, 2), (149, 2), (300, 2))

### Saving reproducible split files

Write each dataframe to a separate CSV. Omitting the pandas index prevents an unrelated numeric column from becoming accidental model
input when the files are loaded later.

In [7]:
# Persist each split without a redundant dataframe-index column.
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

## 6.5 Reusing the GPT-2 tokenizer

Classification inputs must use the same token-to-ID mapping as pretraining. GPT-2's `<|endoftext|>` marker has a dedicated vocabulary
ID when explicitly permitted through `allowed_special`.

The next dataset stage can use this known token as a separator or padding value without expanding `vocab_size`. Encoding returns a Python
list containing one token ID; tensors and batches will be constructed later.

In [8]:
import tiktoken

# Reuse the tokenizer whose vocabulary matches the pretrained GPT-2 weights.
tokenizer = tiktoken.get_encoding("gpt2")
# One special end-of-text marker encodes as (num_tokens=1,).
endoftext_id = tokenizer.encode(
    "<|endoftext|>",
    allowed_special={"<|endoftext|>"},
)
print(endoftext_id)

[50256]


## 6.6 Building a fixed-length classification dataset

PyTorch batches require examples with compatible shapes, but SMS messages contain different token counts. `SpamDataset` therefore:

1. loads one saved split;
2. tokenizes every message with GPT-2;
3. chooses or accepts a shared `max_length`;
4. truncates messages that exceed that length;
5. pads shorter messages with GPT-2's end-of-text token ID; and
6. returns each token-ID tensor with its integer class label.

```text
one input       (num_tokens=max_length,)
one label       ()
complete batch  (batch_size, num_tokens=max_length)
batch labels    (batch_size,)
```

When training, validation, and test datasets are created, they should receive the same explicit `max_length`—normally derived from the
training set and capped at the model's `context_length`. Deriving it independently for each split would produce incompatible shapes and
allow validation or test data to influence preprocessing.

In [9]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset[tuple[torch.Tensor, torch.Tensor]]):
    """Tokenize, truncate, and pad labeled SMS messages from one CSV split."""

    def __init__(
        self,
        csv_file: str | Path,
        tokenizer: tiktoken.Encoding,
        max_length: int | None = None,
        pad_token_id: int = 50256,
    ) -> None:
        """Load one split and prepare fixed-length token-ID sequences.

        Args:
            csv_file: CSV containing `Text` and integer `Label` columns.
            tokenizer: GPT-2 tokenizer used to encode each message.
            max_length: Token IDs retained per message. When `None`, use the
                longest encoded message in this split.
            pad_token_id: Token ID appended to shorter messages. GPT-2 uses
                `50256` for its end-of-text token.
        """
        # data: (num_messages, num_columns=2)
        self.data = pd.read_csv(csv_file)

        # Each inner list initially has its message's natural num_tokens.
        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]

        if max_length is None:
            # Derive one shared length from the longest message in this split.
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncation prevents inputs from exceeding the selected capacity.
            self.encoded_texts = [
                encoded_text[: self.max_length] for encoded_text in self.encoded_texts
            ]

        # Every inner list now has num_tokens=max_length, enabling batching.
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(
        self,
        index: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Return one fixed-length input and its scalar class label.

        Args:
            index: Zero-based row position in this dataset split.

        Returns:
            Input token IDs shaped `(num_tokens=max_length,)` and a scalar
            class-label tensor shaped `()`.

        Raises:
            IndexError: If `index` is outside the dataset.
        """
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        # encoded_tensor: (num_tokens=max_length,); label_tensor: ()
        encoded_tensor = torch.tensor(encoded, dtype=torch.long)
        label_tensor = torch.tensor(label, dtype=torch.long)
        return encoded_tensor, label_tensor

    def __len__(self) -> int:
        """Return the number of labeled messages in this split.

        Returns:
            Number of rows loaded from the CSV file.
        """
        return len(self.data)

    def _longest_encoded_length(self) -> int:
        """Find the largest natural token count in this split.

        Returns:
            Maximum encoded-message length, or `0` for an empty split.
        """
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

## 6.7 Instantiating consistent dataset splits

Create the training dataset first with `max_length=None`. Only training data may determine preprocessing choices; validation and test
sets remain unseen sources of evaluation evidence.

In [10]:
# Derive one preprocessing length from training data only.
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer,
)

### Inspecting the training-derived length

The longest training message contains 120 GPT-2 token IDs. This is well below GPT-2 small's configured `context_length=1_024`, so every
training message fits without exceeding model capacity.

In [11]:
# The longest training message contains this many GPT-2 token IDs.
print("Training max_length:", train_dataset.max_length)

Training max_length: 120


### Applying one length to held-out splits

Pass the training dataset's `max_length` into validation and test datasets. This produces compatible tensors and prevents held-out
messages from determining input shape. Any held-out message longer than 120 tokens is deliberately truncated.

In [12]:
# Reuse the training-derived length so every split has compatible shapes.
# Longer validation or test messages are truncated instead of influencing setup.
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer,
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer,
)

## 6.8 Building classification data loaders

A data loader stacks dataset items into batches:

```text
input_batch   (batch_size, num_tokens)
target_batch  (batch_size,)
```

Shuffle training examples so successive optimizer steps see different class combinations. Keep validation and test order stable because
evaluation does not update parameters.

`drop_last=True` discards the training remainder so every optimizer step has eight examples. It is convenient rather than architecturally
required here. Evaluation uses `drop_last=False` because accuracy must include every held-out example.

In [13]:
from torch.utils.data import DataLoader

# Zero worker subprocesses are reliable in an interactive Windows notebook.
num_workers = 0
batch_size = 8
# Fix the training loader's shuffled row order for reproducibility.
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    # Keep every optimizer step at the configured batch_size.
    drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    # Evaluation must include the final incomplete batch.
    drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False,
)

### Inspecting one complete batch

Read only the first training batch to confirm the dataset–loader contract. The leading dimension contains eight independent messages;
the second contains 120 padded token positions. Labels provide one scalar target per message.

In [14]:
# Read one batch without traversing the complete training loader.
input_batch, target_batch = next(iter(train_loader))
# input_batch: (batch_size=8, num_tokens=120)
# target_batch: (batch_size=8,)
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions:", target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions: torch.Size([8])


### Counting optimizer and evaluation batches

`len(data_loader)` reports how many batches an iteration yields. The training count excludes its incomplete remainder, whereas validation
and test counts include their smaller final batches.

In [15]:
# len(loader) counts batches rather than individual examples.
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
19 validation batches
38 test batches


## 6.9 Loading pretrained GPT-2 weights

Classification fine-tuning should begin with general language representations rather than random parameters. Select a GPT-2 variant,
construct the exact matching `GPTConfig`, download its OpenAI checkpoint, and copy every converted array into the local architecture.

The checkpoint retains its full `context_length=1_024`, although SMS preprocessing uses only 120 token positions. Model capacity and the
current input's `num_tokens` are different concepts.

### Selecting checkpoint dimensions

Each OpenAI GPT-2 size changes embedding width, attention-head count, and Transformer-layer count. Store those values under the canonical
`GPTConfig` identifiers so changing `CHOOSE_MODEL` updates the complete local architecture rather than silently retaining small-model
dimensions.

In [16]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

# Model-specific dimensions use the canonical GPTConfig field names.
model_configs = {
    "gpt2-small (124M)": {
        "emb_dim": 768,
        "num_layers": 12,
        "num_heads": 12,
    },
    "gpt2-medium (355M)": {
        "emb_dim": 1024,
        "num_layers": 24,
        "num_heads": 16,
    },
    "gpt2-large (774M)": {
        "emb_dim": 1280,
        "num_layers": 36,
        "num_heads": 20,
    },
    "gpt2-xl (1558M)": {
        "emb_dim": 1600,
        "num_layers": 48,
        "num_heads": 25,
    },
}

### Reconstructing and loading the selected model

The download helper supplies exact vocabulary and context metadata plus converted parameter arrays. Combine that metadata with the
selected model dimensions, validate it through `GPTConfig`, instantiate `GPTModel`, and copy the checkpoint arrays.

At this point the model remains on CPU. A later training section can move it and each batch to the same CUDA device.

In [17]:
from gpt_download import download_and_load_gpt2

from build_llms_from_scratch_companion.model import GPTConfig, GPTModel
from build_llms_from_scratch_companion.training import load_weights_into_gpt

# Extract the helper's checkpoint-size identifier, such as "124M".
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2",
)
# Combine exact checkpoint metadata with the selected variant dimensions.
cfg = GPTConfig(
    vocab_size=settings["n_vocab"],
    context_length=settings["n_ctx"],
    dropout_rate=0.0,
    qkv_bias=True,
    **model_configs[CHOOSE_MODEL],
)
model = GPTModel(cfg)
load_weights_into_gpt(model, params)
# Disable dropout for deterministic baseline generation.
model.eval()
print("Loaded pretrained model:", CHOOSE_MODEL)

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe
Loaded pretrained model: gpt2-small (124M)


## 6.10 Verifying ordinary pretrained generation

Before changing the architecture, generate a short continuation as a sanity check. Coherent output confirms that configuration, weight
mapping, tokenization, and the language-model output head agree.

In [18]:
from build_llms_from_scratch_companion.generation import generate_text_simple
from build_llms_from_scratch_companion.training import (
    text_to_token_ids,
    token_ids_to_text,
)

# Tokenization creates CPU IDs; place them beside the model parameters.
model_device = next(model.parameters()).device
# input_ids: (batch_size=1, num_tokens=3)
input_ids = text_to_token_ids(INPUT_PROMPT, tokenizer).to(model_device)
# token_ids: (batch_size=1, num_tokens=3 + max_new_tokens=15)
token_ids = generate_text_simple(
    model=model,
    idx=input_ids,
    max_new_tokens=15,
    context_size=cfg.context_length,
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves forward, but it's not enough.

"I'm not going


## 6.11 Testing the base model as a prompted classifier

Ask base GPT-2 for a yes/no spam decision before fine-tuning. The model instead continues patterns from the prompt because it was trained
for next-token prediction, not instruction following.

This is a qualitative baseline, not predictive-performance measurement. Accuracy becomes measurable only after a classification output,
labeled batches, and an evaluation function are connected.

In [19]:
spam_prompt = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
# prompt_ids: (batch_size=1, num_tokens)
prompt_ids = text_to_token_ids(spam_prompt, tokenizer).to(model_device)
token_ids = generate_text_simple(
    model=model,
    idx=prompt_ids,
    max_new_tokens=23,
    context_size=cfg.context_length,
)
print(token_ids_to_text(token_ids, tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner


## 6.12 Freezing pretrained parameters

Set every existing parameter's `requires_grad` flag to `False`. Forward passes still use all pretrained weights, but backpropagation does
not calculate or store their gradients. This reduces optimization cost and protects general language representations while the new
classifier starts from random values.

Freezing is applied before replacing the output head, so the subsequently created head remains trainable by default.

In [ ]:
# Freeze embeddings, all Transformer blocks, normalization, and the old head.
for parameter in model.parameters():
    parameter.requires_grad = False

## 6.13 Replacing vocabulary prediction with class prediction

The pretrained output head maps each token representation from `emb_dim=768` to `vocab_size=50_257` next-token logits. Classification
instead needs two scores—one for ham and one for spam—so replace only that projection:

```text
before  (batch_size, num_tokens, vocab_size)
after   (batch_size, num_tokens, num_classes=2)
```

The new linear layer is randomly initialized and trainable. Moving it to the model's current device prevents a CPU/CUDA mismatch if the
base model was moved before this cell is rerun.

In [ ]:
# Reproduce the random initialization of the new classification head.
torch.manual_seed(123)
num_classes = 2
model_device = next(model.parameters()).device
# Map each emb_dim representation to two class logits on the model's device.
model.out_head = torch.nn.Linear(
    in_features=cfg.emb_dim,
    out_features=num_classes,
).to(model_device)

## 6.14 Partially fine-tuning the representation

Training only the new head would be a **linear probe**: it can recombine existing GPT-2 features but cannot change those features for
spam detection. That is a valid, cheaper baseline and would be worth measuring first.

The book additionally unfreezes the final Transformer block and final layer normalization. This gives the model limited capacity to
adapt high-level representations while keeping the earlier eleven blocks fixed. It is a compromise between:

- **head-only training:** cheapest and least prone to overfitting;
- **partial fine-tuning:** more adaptable with moderate cost; and
- **full fine-tuning:** most adaptable but most expensive and easiest to overfit.

So your instinct is valid: retraining only the output head is possible. The final block and normalization are unfrozen because the
pretrained model was never taught that its representations should separate ham from spam.

In [ ]:
# Adapt only the highest Transformer representations to classification.
for parameter in model.trf_blocks[-1].parameters():
    parameter.requires_grad = True
# Let the final normalization scale and shift adapt with those representations.
for parameter in model.final_norm.parameters():
    parameter.requires_grad = True

## 6.15 Inspecting the untrained classification output

Run one short message through the modified architecture before classification training. This checks tensor shapes, not predictive
performance. The classification head is still random, so its scores and implied class decision are not meaningful.

A causal GPT produces one representation per input position. The classifier therefore emits two class logits at every position, and the
next step selects the final position because it has attended to all preceding message tokens.

In [ ]:
# Tokenization returns four CPU IDs; move the complete batch to the model.
input_ids = tokenizer.encode("Do you have time")
# inputs: (batch_size=1, num_tokens=4)
inputs = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(model_device)
print("Inputs:", inputs)
print("Input dimensions:", inputs.shape)

In [ ]:
# This is an inference-only shape check, so no gradients are required.
with torch.no_grad():
    # logits: (batch_size=1, num_tokens=4, num_classes=2)
    logits = model(inputs)
print("Class logits:", logits)
print("Class-logit dimensions:", logits.shape)

In [ ]:
# The final position has processed every preceding token in the message.
# last_token_logits: (batch_size=1, num_classes=2)
last_token_logits = logits[:, -1, :]
print("Last-position class logits:", last_token_logits)

## Chapter 6 summary

The chapter now prepares a pretrained GPT-2 model for supervised classification:

- tokenize and batch SMS messages with training-derived preprocessing;
- load matching OpenAI language-model weights;
- freeze pretrained parameters before replacing the vocabulary head;
- map each token representation to two class logits;
- unfreeze only the final Transformer block and final normalization; and
- select final-position logits whose representation has processed the complete message.

The current class scores are random because no supervised update has occurred. Predictive performance becomes meaningful only after
classification loss, optimization, and held-out accuracy evaluation are implemented.